# Eats Express: local CSV processing

## 1. Import dependencies

In [ ]:
# Standard library imports, followed by data-processing libraries.
import re

import apache_beam as beam
import pandas as pd


## 2. Load and inspect the source data

Paths below are relative to the notebooks directory.

In [ ]:
# Read data
df = pd.read_csv("../data/food_daily.csv")
df.head()

In [ ]:
# Confirm the source column names and order.
df.columns

In [ ]:
# Inspect missing values and inferred data types.
df.info()

## 3. Define row-cleaning helpers

The items field is column 4 and delivery status is column 8 (zero-based).

In [ ]:
def remove_last_colon(item):
    """Remove a trailing colon from the items field, when present."""
    if item.endswith(':'):
        return item[: -1]
    return item

def process_row(row):
    """Clean the items column and return the comma-separated row."""
    columns = row.split(',')
    if len(columns) > 4:
        columns[4] = remove_last_colon(columns[4])
    return ','.join(columns)

def remove_special_characters(row):
    """Keep letters, digits, and spaces within each comma-separated field."""
    columns = row.split(',')
    ret = ''
    for col in columns:
        ret += re.sub(r'[^a-zA-Z0-9 ]', '', col) + ','
    return ret[: -1]

def print_row(row):
    """Print a row when inspecting intermediate pipeline output."""
    print(row)


## 4. Run the local Beam pipeline

Write delivered and not-delivered orders to separate text outputs. On-hold and cancelled orders are excluded.

In [ ]:
input_file = '../data/food_daily.csv'
output_path = '../data/output/processed'
# Execute locally and wait for both output branches to finish.
with beam.Pipeline() as p:
    # Apply the same cleaning steps before splitting by delivery status.
    cleaned_data = ( 
          p 
          | "Read Input File" >> beam.io.ReadFromText(input_file,skip_header_lines = 1)
          | "Process Items Column" >> beam.Map(process_row)
          | "Convert to lowercase" >> beam.Map(lambda row: row.lower())
          | "Remove special characters" >> beam.Map(remove_special_characters)
    )

    delivered_orders = (
        cleaned_data
        | 'Filter delivered data' >> beam.Filter(lambda row: row and row.split(',')[8].strip().lower() == 'delivered')
        | 'Write Delivered File' >> beam.io.WriteToText(output_path + '/delivered')
    )

    undelivered_orders = (
        cleaned_data
         | 'Filter undeliver data' >> beam.Filter(lambda row: row and row.split(',')[8].strip().lower() == 'not delivered')
         | 'Write Undelivered File' >> beam.io.WriteToText(output_path + '/undelivered')
        )